In [1]:
# Cell 0: Install dependencies (run once in VSCode Jupyter)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

0

# Azalyst Alpha Research Engine v2.1 — LOCAL GPU (RTX 2050)

**Built by Azalyst Research** | Cross-Sectional Crypto Alpha with XGBoost CUDA

### Local GPU Configuration
- **GPU**: NVIDIA RTX 2050 (4GB GDDR6) — CUDA pinned to device 0
- **CPU**: Intel i5-11260H (6C/12T)  
- **VRAM Guard**: 2M rows max (prevents OOM on 4GB)
- **Dual CUDA API**: Supports both `device='cuda'` (new) and `tree_method='gpu_hist'` (old)

### Three Pillars of Alpha (v2.1)
1. **Fractional Differentiation** — Memory-preserving stationarity (AFML Ch. 5)
2. **Meta-Labeling** — Confidence-weighted position sizing (AFML Ch. 3)
3. **IC-Weighted Signal Combination** — Dynamic factor reweighting (Grinold & Kahn)

**56 Features** → XGBoost CUDA → Purged K-Fold CV → Walk-Forward Year 3 → Position-Tracked Fees

In [2]:
# ── Cell 1: Imports & Paths ───────────────────────────────────────────────────
import os, sys, time, json, gc, pickle, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
from scipy import stats
warnings.filterwarnings('ignore')

DATA_DIR          = './data'
RESULTS_DIR       = './results'
FEATURE_STORE_DIR = './feature_cache'

for d in [RESULTS_DIR, f'{RESULTS_DIR}/models', f'{RESULTS_DIR}/shap', FEATURE_STORE_DIR]:
    os.makedirs(d, exist_ok=True)

parquet_files = sorted(Path(DATA_DIR).rglob('*.parquet'))
if not parquet_files:
    parquet_files = sorted(Path(DATA_DIR).parent.rglob('*.parquet'))
    if parquet_files: DATA_DIR = str(parquet_files[0].parent)

print(f'DATA_DIR          : {DATA_DIR}')
print(f'RESULTS_DIR       : {RESULTS_DIR}')
print(f'FEATURE_STORE_DIR : {FEATURE_STORE_DIR}')
print(f'Parquet files found: {len(parquet_files)}')

DATA_DIR          : ./data
RESULTS_DIR       : ./results
FEATURE_STORE_DIR : ./feature_cache
Parquet files found: 443


In [3]:
# ── Cell 2: GPU Setup (RTX 2050 — Dual CUDA API) ──────────────────────────────
import subprocess, xgboost as xgb

# Pin to RTX 2050 (device 0) — ignore Intel UHD integrated GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"]    = "PCI_BUS_ID"

try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
    print(r.stdout.decode()[:300])
except: pass


def detect_cuda_api():
    """
    Returns 'new' if device='cuda' works (XGBoost 2.0+),
    'old' if tree_method='gpu_hist' works, else None (CPU fallback).
    """
    try:
        _x = np.random.rand(1000, 10).astype(np.float32)
        _y = np.random.randint(0, 2, 1000)
        try:
            xgb.XGBClassifier(device='cuda', n_estimators=5, verbosity=0).fit(_x, _y)
            return 'new'
        except:
            pass
        try:
            xgb.XGBClassifier(tree_method='gpu_hist', n_estimators=5, verbosity=0).fit(_x, _y)
            return 'old'
        except:
            return None
    except:
        return None


CUDA_API = detect_cuda_api()
HAS_GPU  = CUDA_API is not None

import psutil
def mem_gb():
    return psutil.virtual_memory().used / 1e9

def aggressive_gc():
    gc.collect()

print(f'XGBoost {xgb.__version__}: CUDA_API={CUDA_API} | HAS_GPU={HAS_GPU}')

Thu Mar 19 12:52:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.79                 Driver Version: 595.79         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+-------------
XGBoost 3.2.0: CUDA_API=new | HAS_GPU=True


In [4]:
# ── Cell 3: System Diagnostics ───────────────────────────────────────────────
import psutil, platform
print(f'Platform  : {platform.system()} {platform.release()}')
print(f'CPU       : {platform.processor()}')
print(f'RAM       : {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'GPU       : {"RTX 2050 CUDA" if HAS_GPU else "CPU only"}')
print(f'CUDA API  : {CUDA_API or "none"}')
print(f'XGBoost   : {xgb.__version__}')
print(f'VRAM cap  : 2,000,000 rows')

Platform  : Windows 10
CPU       : Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
RAM       : 16.9 GB
GPU       : RTX 2050 CUDA
CUDA API  : new
XGBoost   : 3.2.0
VRAM cap  : 2,000,000 rows


In [5]:
# -- Cell 3: Configuration ---------------------------------------------------
import pickle

FEATURE_COLS = [
    'ret_1bar','ret_1h','ret_4h','ret_1d','ret_2d','ret_3d','ret_1w',
    'vol_ratio','vol_ret_1h','vol_ret_1d','obv_change','vpt_change','vol_momentum',
    'rvol_1h','rvol_4h','rvol_1d','vol_ratio_1h_1d','atr_norm','parkinson_vol','garman_klass',
    'rsi_14','rsi_6','macd_hist','bb_pos','bb_width','stoch_k','stoch_d','cci_14','adx_14','dmi_diff',
    'vwap_dev','amihud','kyle_lambda','spread_proxy','body_ratio','candle_dir',
    'wick_top','wick_bot','price_accel','skew_1d','kurt_1d','max_ret_4h',
    'wq_alpha001','wq_alpha012','wq_alpha031','wq_alpha098',
    'cs_momentum','cs_reversal','vol_adjusted_mom','trend_consistency',
    'vol_regime','trend_strength','corr_btc_proxy','hurst_exp','fft_strength',
    'frac_diff_close',
]

RETRAIN_WEEKS   = 13          # quarterly
TOP_QUANTILE    = 0.15        # top/bottom 15% for long/short
FEE_RATE        = 0.001       # 0.1% per leg
ROUND_TRIP_FEE  = FEE_RATE * 2
STOP_LOSS_PCT   = -2.0
TAKE_PROFIT_PCT = 5.0
HORIZON_BARS    = 48          # 4H @ 5-min bars
MAX_TRAIN_ROWS  = 2_000_000   # RTX 2050 4GB VRAM guard — do NOT raise above 2M

print(f'[Cell 3] Config loaded.')
print(f'  Features      : {len(FEATURE_COLS)}')
print(f'  GPU mode      : {HAS_GPU}')
print(f'  Retrain every : {RETRAIN_WEEKS} weeks')
print(f'  Fee round-trip: {ROUND_TRIP_FEE*100:.2f}%')

[Cell 3] Config loaded.
  Features      : 56
  GPU mode      : True
  Retrain every : 13 weeks
  Fee round-trip: 0.20%


In [6]:
# -- Cell 4: Feature Engineering v2.1 (56 features + frac_diff) ---------------
BARS_PER_HOUR = 12
BARS_PER_DAY  = 288

def _rsi(s, n):
    d = s.diff(); g = d.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    ls = (-d).clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100 / (1 + g / ls.replace(0, np.nan))

def _ema(s, n):
    return s.ewm(span=n, adjust=False).mean()


def frac_diff_ffd(series, d=0.4, threshold=1e-5):
    """
    Fixed-Width Window Fractional Differentiation (AFML Ch. 5).
    d in (0,1): d=0 is raw price (non-stationary, max memory),
    d=1.0 is standard returns (stationary, zero memory).
    d~0.4 balances stationarity with memory retention.
    """
    w = [1.0]
    k = 1
    while True:
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < threshold:
            break
        w.append(w_k)
        k += 1
    w = np.array(w[::-1], dtype=np.float64)
    width = len(w)
    arr = series.values.astype(np.float64)
    out = np.full(len(arr), np.nan, dtype=np.float64)
    for i in range(width - 1, len(arr)):
        chunk = arr[i - width + 1 : i + 1]
        if np.isnan(chunk).any():
            continue
        out[i] = np.dot(w, chunk)
    return pd.Series(out, index=series.index, dtype=np.float32)


def build_features(df, timeframe='5min'):
    """Vectorized feature builder -- 56 features (v2.1 with frac_diff_close)."""
    bph, bpd = BARS_PER_HOUR, BARS_PER_DAY
    if timeframe != '5min':
        mins = {'1h':60,'4h':240,'1d':1440}.get(timeframe, 5)
        bph = max(1, 60 // mins); bpd = max(1, 1440 // mins)
    o, h, l, c, v = df['open'], df['high'], df['low'], df['close'], df['volume']
    f = pd.DataFrame(index=df.index)

    # Returns (7)
    f['ret_1bar'] = c.pct_change()
    for name, bars in [('ret_1h',bph),('ret_4h',bph*4),('ret_1d',bpd),
                       ('ret_2d',bpd*2),('ret_3d',bpd*3),('ret_1w',bpd*7)]:
        f[name] = c.pct_change(bars)

    # Volume (6)
    avg_v = v.rolling(bpd, min_periods=1).mean()
    f['vol_ratio'] = v / avg_v.replace(0, np.nan)
    f['vol_ret_1h'] = v.pct_change(bph)
    f['vol_ret_1d'] = v.pct_change(bpd)
    f['obv_change'] = (v * np.sign(c.diff())).cumsum().pct_change(bph)
    pt = v * c.pct_change()
    f['vpt_change'] = pt.cumsum().pct_change(bph)
    f['vol_momentum'] = v.rolling(bph).mean() / v.rolling(bpd).mean().replace(0, np.nan)

    # Volatility (7)
    f['rvol_1h']  = c.pct_change().rolling(bph).std()
    f['rvol_4h']  = c.pct_change().rolling(bph*4).std()
    f['rvol_1d']  = c.pct_change().rolling(bpd).std()
    f['vol_ratio_1h_1d'] = f['rvol_1h'] / f['rvol_1d'].replace(0, np.nan)
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    f['atr_norm'] = tr.rolling(bph*4).mean() / c.replace(0, np.nan)
    hl = np.log(h / l.replace(0, np.nan))
    co = np.log(c / o.replace(0, np.nan))
    f['parkinson_vol'] = hl.rolling(bpd).apply(lambda x: np.sqrt((x**2).mean() / (4*np.log(2))), raw=True)
    f['garman_klass']  = (0.5 * hl**2 - (2*np.log(2)-1) * co**2).rolling(bpd).mean()

    # Technical (10)
    f['rsi_14'] = _rsi(c, 14) / 100
    f['rsi_6']  = _rsi(c, 6) / 100
    e12, e26 = _ema(c,bph*12), _ema(c,bph*26)
    macd_line = e12 - e26; signal_line = _ema(macd_line, 9*bph)
    f['macd_hist'] = (macd_line - signal_line) / c.replace(0, np.nan)
    bb_mid = c.rolling(20*bph).mean(); bb_std = c.rolling(20*bph).std()
    f['bb_pos']   = (c - bb_mid) / (2 * bb_std.replace(0, np.nan))
    f['bb_width'] = (4 * bb_std) / bb_mid.replace(0, np.nan)
    lo_n, hi_n = l.rolling(14*bph).min(), h.rolling(14*bph).max()
    f['stoch_k'] = ((c - lo_n) / (hi_n - lo_n).replace(0, np.nan))
    f['stoch_d'] = f['stoch_k'].rolling(3*bph).mean()
    typical = (h + l + c) / 3
    f['cci_14'] = (typical - typical.rolling(14*bph).mean()) / (0.015 * typical.rolling(14*bph).std().replace(0, np.nan))
    pdi = ((h.diff().clip(lower=0)).ewm(span=14*bph).mean() / tr.ewm(span=14*bph).mean().replace(0, np.nan)) * 100
    ndi = ((-l.diff()).clip(lower=0).ewm(span=14*bph).mean() / tr.ewm(span=14*bph).mean().replace(0, np.nan)) * 100
    f['adx_14']  = (abs(pdi - ndi) / (pdi + ndi).replace(0, np.nan)).ewm(span=14*bph).mean()
    f['dmi_diff'] = (pdi - ndi) / 100

    # Microstructure (6)
    vwap = (typical * v).cumsum() / v.cumsum().replace(0, np.nan)
    f['vwap_dev'] = (c - vwap) / vwap.replace(0, np.nan)
    abs_r = c.pct_change().abs()
    f['amihud'] = abs_r / (v * c).replace(0, np.nan)
    f['kyle_lambda'] = abs_r.rolling(bph).mean() / v.rolling(bph).mean().replace(0, np.nan)
    f['spread_proxy'] = (h - l) / c.replace(0, np.nan)
    body = abs(c - o)
    f['body_ratio'] = body / (h - l).replace(0, np.nan)
    f['candle_dir'] = np.sign(c - o)

    # Price structure (6)
    f['wick_top'] = (h - pd.concat([o, c], axis=1).max(axis=1)) / (h - l).replace(0, np.nan)
    f['wick_bot'] = (pd.concat([o, c], axis=1).min(axis=1) - l) / (h - l).replace(0, np.nan)
    f['price_accel'] = c.pct_change().diff()
    f['skew_1d'] = c.pct_change().rolling(bpd).skew()
    f['kurt_1d'] = c.pct_change().rolling(bpd).kurt()
    f['max_ret_4h'] = c.pct_change().rolling(bph*4).max()

    # WQ Alphas (4)
    f['wq_alpha001'] = (-c.pct_change().rolling(bph).std().rank(pct=True)).fillna(0)
    f['wq_alpha012'] = (-np.sign(v.diff()) * c.pct_change()).fillna(0)
    delay_c = c.shift(bph)
    f['wq_alpha031'] = (
        (c - delay_c).rank(pct=True).rolling(bph*4).mean() * (-c.pct_change(bph*2)) * v.pct_change(bph)
    ).fillna(0)
    f['wq_alpha098'] = (
        c.rolling(5*bph).mean().rank(pct=True) - c.pct_change(bph*2) * v.rank(pct=True)
    ).fillna(0)

    # Cross-section + Regime (8)
    f['cs_momentum']      = c.pct_change(bpd*5).rank(pct=True)
    f['cs_reversal']      = (-c.pct_change(bpd)).rank(pct=True)
    f['vol_adjusted_mom'] = f['ret_1w'] / f['rvol_1d'].replace(0, np.nan)
    r5 = c.pct_change(bpd*5)
    sign_same = (r5.rolling(bpd*5, min_periods=bpd).apply(lambda x: (np.sign(x)==np.sign(x.iloc[-1])).mean(), raw=False))
    f['trend_consistency'] = sign_same
    f['vol_regime']     = f['rvol_1d'].rolling(bpd*5).rank(pct=True)
    f['trend_strength'] = abs(f['ret_1w']) / f['rvol_1d'].replace(0, np.nan)
    btc_proxy = c.rolling(bpd).corr(c.rolling(bpd).mean())
    f['corr_btc_proxy'] = btc_proxy

    # Hurst exponent (simplified R/S)
    hurst_win = bpd * 5
    hurst_exp = pd.Series(np.nan, index=df.index, dtype=np.float32)
    close_arr = c.values.astype(np.float64)
    for i in range(hurst_win, len(close_arr)):
        seg = close_arr[i-hurst_win:i]
        rs_vals = []
        for ss in [hurst_win//4, hurst_win//2, hurst_win]:
            sub = np.diff(np.log(seg[-ss:] + 1e-10))
            mean_sub = np.mean(sub)
            cumdev = np.cumsum(sub - mean_sub)
            R = np.max(cumdev) - np.min(cumdev)
            S = np.std(sub) + 1e-10
            rs_vals.append(R / S)
        if len(rs_vals) >= 2 and all(r > 0 for r in rs_vals):
            ns = [hurst_win//4, hurst_win//2, hurst_win]
            log_n = np.log(ns[:len(rs_vals)])
            log_rs = np.log(rs_vals)
            if len(log_n) > 1:
                slope = np.polyfit(log_n, log_rs, 1)[0]
                hurst_exp.iloc[i] = float(slope)
    f['hurst_exp'] = hurst_exp

    # FFT strength
    fft_win = bpd * 2
    fft_strength = pd.Series(np.nan, index=df.index, dtype=np.float32)
    for i in range(fft_win, len(close_arr)):
        seg = close_arr[i-fft_win:i]
        seg_ret = np.diff(np.log(seg + 1e-10))
        fft_mag = np.abs(np.fft.rfft(seg_ret))
        fft_strength.iloc[i] = float(np.max(fft_mag[1:]) / (np.mean(fft_mag[1:]) + 1e-10))
    f['fft_strength'] = fft_strength.astype(np.float32)

    # Fractional differentiation of log-price (AFML Ch. 5)
    f['frac_diff_close'] = frac_diff_ffd(np.log(c.clip(lower=1e-10)), d=0.4)

    return f.replace([np.inf, -np.inf], np.nan).shift(1).astype(np.float32)

print('[Cell 4] Feature Engineering v2.1 defined (56 features + frac_diff_ffd).')
print(f'  Includes: frac_diff_ffd() -- AFML Ch. 5 memory-preserving differencing')

[Cell 4] Feature Engineering v2.1 defined (56 features + frac_diff_ffd).
  Includes: frac_diff_ffd() -- AFML Ch. 5 memory-preserving differencing


In [7]:
# ── Cell 5: Feature Store ────────────────────────────────────────────────────
def build_feature_store(parquet_files):
    for f in parquet_files:
        out_path = Path(FEATURE_STORE_DIR) / f.name
        if out_path.exists(): continue
        try:
            df = pd.read_parquet(f)
            df.index = pd.to_datetime(df.index, utc=True)
            feat = build_features(df)
            feat['future_ret'] = np.log(df['close'].shift(-HORIZON_BARS) / df['close'])
            feat['symbol'] = f.stem
            feat.dropna(subset=['future_ret'] + FEATURE_COLS, how='all').to_parquet(out_path)
        except: continue

In [8]:
# -- Cell 6: Build Feature Store (run once, skip if cache exists) -----------
print('[Cell 6] Building feature store...')
t0 = time.time()

existing = list(Path(FEATURE_STORE_DIR).glob('*.parquet'))
if len(existing) >= len(parquet_files) * 0.9:
    print(f'  Cache already complete: {len(existing)} files -- skipping rebuild')
else:
    build_feature_store(parquet_files)
    existing = list(Path(FEATURE_STORE_DIR).glob('*.parquet'))
    print(f'  Feature store complete: {len(existing)} symbols in {time.time()-t0:.1f}s')

sample = existing[0] if existing else None
if sample:
    test_df = pd.read_parquet(sample)
    print(f'  Sample: {sample.stem} | shape={test_df.shape}')
    missing = [c for c in FEATURE_COLS if c not in test_df.columns]
    print(f'  Missing feat cols: {missing if missing else "none -- all present"}')

gc.collect()


[Cell 6] Building feature store...
  Cache already complete: 443 files -- skipping rebuild
  Sample: 0GUSDT | shape=(47779, 57)
  Missing feat cols: ['frac_diff_close']


0

In [ ]:
# -- Cell 7: Load Feature Store & Build Train/Test Split ---------------------
print('[Cell 7] Loading feature store...')
t0 = time.time()

all_files = sorted(Path(FEATURE_STORE_DIR).glob('*.parquet'))
print(f'  Found {len(all_files)} cached symbol files')

frames = []
for fp in all_files:
    try:
        df = pd.read_parquet(fp)
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index, utc=True)
        elif df.index.tz is None:
            df.index = df.index.tz_localize('UTC')
        if 'symbol' not in df.columns:
            df['symbol'] = fp.stem
        frames.append(df)
    except Exception as e:
        print(f'  [WARN] {fp.stem}: {e}')

if not frames:
    raise RuntimeError('No feature files loaded. Re-run Cell 6.')

ALL_DATA = pd.concat(frames, axis=0).sort_index()
print(f'  Pooled: {len(ALL_DATA):,} rows | {ALL_DATA["symbol"].nunique()} symbols')
print(f'  Range : {ALL_DATA.index.min().date()} to {ALL_DATA.index.max().date()}')

# Year split: Year1+2 = train, Year3 = test (never touched during training)
global_min = ALL_DATA.index.min()
global_max = ALL_DATA.index.max()
total_span = global_max - global_min
YEAR2_END  = global_min + (total_span * 2 / 3)
YEAR3_START = YEAR2_END
print(f'\n  Train (Y1+Y2): {global_min.date()} to {YEAR2_END.date()}')
print(f'  Test  (Y3)  : {YEAR3_START.date()} to {global_max.date()}')

# Build training matrix (Year 1+2 only)
MAX_TRAIN_ROWS = 2_000_000  # RTX 2050 4GB VRAM guard — do NOT raise above 2M
train_df = ALL_DATA[ALL_DATA.index < YEAR2_END].copy()

# Cross-sectional median label: did this bar outperform median at t+HORIZON_BARS?
if 'alpha_label' not in train_df.columns:
    if 'future_ret' in train_df.columns:
        train_df['alpha_label'] = (
            train_df.groupby(train_df.index)['future_ret']
            .transform(lambda x: (x > x.median()).astype(float))
        )
    else:
        raise KeyError('future_ret not found in feature store. Re-run Cell 6.')

# Fill any missing feature cols with 0
for c in FEATURE_COLS:
    if c not in train_df.columns:
        train_df[c] = 0.0

feat_matrix = train_df[FEATURE_COLS].values.astype(np.float32)
label_arr   = train_df['alpha_label'].values.astype(np.float32)
ret_arr     = train_df['future_ret'].values.astype(np.float32)

valid_mask  = np.isfinite(feat_matrix).all(axis=1) & np.isfinite(label_arr)
feat_matrix = feat_matrix[valid_mask]
label_arr   = label_arr[valid_mask]
ret_arr     = ret_arr[valid_mask]

if len(feat_matrix) > MAX_TRAIN_ROWS:
    idx = np.random.choice(len(feat_matrix), MAX_TRAIN_ROWS, replace=False)
    idx.sort()
    feat_matrix, label_arr, ret_arr = feat_matrix[idx], label_arr[idx], ret_arr[idx]

X_TRAIN, Y_TRAIN, Y_RET = feat_matrix, label_arr, ret_arr
print(f'\n  Training matrix: {X_TRAIN.shape[0]:,} rows x {X_TRAIN.shape[1]} features')
print(f'  Label balance  : {Y_TRAIN.mean()*100:.1f}% positive (target ~50%)')
print(f'  Loaded in      : {time.time()-t0:.1f}s')
del train_df, frames
gc.collect()


[Cell 7] Loading feature store...
  Found 443 cached symbol files


In [ ]:
# -- Cell 8: PurgedTimeSeriesCV + train_model + train_meta_model ---------------
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
from scipy import stats as sp_stats

class PurgedTimeSeriesCV:
    '''Purged K-Fold with 48-bar embargo -- prevents leakage from overlapping windows.'''
    def __init__(self, n_splits=5, gap=48):
        self.n_splits = n_splits
        self.gap = gap
    def split(self, X):
        n = len(X)
        fold_size = n // (self.n_splits + 1)
        for i in range(self.n_splits):
            train_end = (i + 1) * fold_size
            val_start = train_end + self.gap
            val_end   = val_start + fold_size
            if val_end > n: break
            yield np.arange(0, train_end), np.arange(val_start, val_end)

def compute_ic(y_pred, y_true):
    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() < 10: return 0.0
    return float(sp_stats.spearmanr(y_pred[mask], y_true[mask])[0])

def make_xgb_params(cuda_api):
    p = dict(
        n_estimators=1000, learning_rate=0.02, max_depth=6,
        min_child_weight=30, subsample=0.8, colsample_bytree=0.7,
        colsample_bylevel=0.7, reg_alpha=0.1, reg_lambda=1.0,
        eval_metric='auc', early_stopping_rounds=50,
        verbosity=0, random_state=42,
    )
    if cuda_api == 'new':
        p['device'] = 'cuda'
    elif cuda_api == 'old':
        p['tree_method'] = 'gpu_hist'
    return p

def train_model(X, y, y_ret=None, label='', use_gpu=False):
    '''
    Train XGBoost with Purged K-Fold CV + RobustScaler.
    Returns (model, scaler, importance_series, mean_auc, mean_ic, icir)
    '''
    scaler = RobustScaler()
    Xs = scaler.fit_transform(X)
    cv = PurgedTimeSeriesCV(n_splits=5, gap=48)
    aucs, ics = [], []
    for fold, (tr, val) in enumerate(cv.split(Xs), 1):
        if len(np.unique(y[val])) < 2: continue
        m = xgb.XGBClassifier(**make_xgb_params(CUDA_API))
        m.fit(Xs[tr], y[tr], eval_set=[(Xs[val], y[val])], verbose=False)
        probs = m.predict_proba(Xs[val])[:, 1]
        try: aucs.append(roc_auc_score(y[val], probs))
        except: pass
        if y_ret is not None and np.isfinite(y_ret[val]).any():
            ics.append(compute_ic(probs, y_ret[val]))
    mean_auc = float(np.mean(aucs)) if aucs else 0.0
    mean_ic  = float(np.mean(ics))  if ics  else 0.0
    icir     = float(np.mean(ics) / (np.std(ics) + 1e-8)) if len(ics) > 1 else 0.0
    final = xgb.XGBClassifier(**make_xgb_params(CUDA_API))
    split = int(len(Xs) * 0.9)
    final.fit(Xs[:split], y[:split], eval_set=[(Xs[split:], y[split:])], verbose=False)
    importance = pd.Series(final.feature_importances_, index=FEATURE_COLS, name='importance').sort_values(ascending=False)
    return final, scaler, importance, mean_auc, mean_ic, icir


def train_meta_model(primary_model, primary_scaler, X, y, label='meta', use_gpu=False):
    """
    Train a second-stage model that predicts P(primary model is correct).
    Uses purged CV to generate honest OOS predictions, avoiding information leakage.
    Output is used for confidence-weighted position sizing (AFML Ch. 3).
    """
    Xs = primary_scaler.transform(X)

    # Collect OOS predictions from purged CV
    cv = PurgedTimeSeriesCV(n_splits=5, gap=48)
    oos_preds = np.full(len(y), np.nan, dtype=np.float32)
    for fold, (tr, val) in enumerate(cv.split(Xs), 1):
        if len(np.unique(y[val])) < 2: continue
        m_temp = xgb.XGBClassifier(**make_xgb_params(CUDA_API))
        m_temp.fit(Xs[tr], y[tr], eval_set=[(Xs[val], y[val])], verbose=False)
        oos_preds[val] = m_temp.predict_proba(Xs[val])[:, 1]

    valid = np.isfinite(oos_preds)
    if valid.sum() < 200:
        print(f'  [{label}] Insufficient OOS data ({valid.sum()} rows) -- skipping meta')
        return None, None

    # Meta-label: 1 if predicted direction matches actual alpha_label
    meta_y = ((oos_preds[valid] >= 0.5).astype(float) == y[valid]).astype(float)

    # Augmented features: scaled X + primary OOS probability
    X_meta = np.column_stack([Xs[valid], oos_preds[valid]])
    meta_scaler = RobustScaler()
    X_meta_s = meta_scaler.fit_transform(X_meta)

    # Shallower XGBoost for meta-model (less overfitting)
    meta_params = make_xgb_params(CUDA_API)
    meta_params.update(n_estimators=500, max_depth=4, min_child_weight=50)
    meta = xgb.XGBClassifier(**meta_params)
    split = int(len(X_meta_s) * 0.9)
    meta.fit(X_meta_s[:split], meta_y[:split],
             eval_set=[(X_meta_s[split:], meta_y[split:])], verbose=False)

    val_acc = float((meta.predict(X_meta_s[split:]) == meta_y[split:]).mean())
    print(f'  [{label}] Meta-model accuracy: {val_acc*100:.1f}% on {valid.sum():,} OOS rows')
    return meta, meta_scaler


print('[Cell 8] PurgedTimeSeriesCV + train_model + train_meta_model defined.')
print(f'  GPU available: {HAS_GPU}  (CUDA_API={CUDA_API})')
print(f'  Meta-labeling: enabled (AFML Ch. 3)')

In [ ]:
# -- Cell 9: Train Base Model + Meta Model (Year 1 + Year 2) ------------------
BASE_MODEL_PATH  = f'{RESULTS_DIR}/models/model_base_y1y2.json'
BASE_SCALER_PATH = f'{RESULTS_DIR}/models/scaler_base_y1y2.pkl'
META_MODEL_PATH  = f'{RESULTS_DIR}/models/meta_model_base.pkl'
META_SCALER_PATH = f'{RESULTS_DIR}/models/meta_scaler_base.pkl'

# --- Primary model ---
if Path(BASE_MODEL_PATH).exists() and Path(BASE_SCALER_PATH).exists():
    print('[Cell 9] Loading cached base model...')
    BASE_MODEL = xgb.XGBClassifier()
    BASE_MODEL.load_model(BASE_MODEL_PATH)
    with open(BASE_SCALER_PATH, 'rb') as f:
        BASE_SCALER = pickle.load(f)
    print(f'  Loaded: {BASE_MODEL_PATH}')
else:
    print(f'[Cell 9] Training base model on {len(X_TRAIN):,} rows  (GPU={HAS_GPU})...')
    t0 = time.time()
    BASE_MODEL, BASE_SCALER, importance, mean_auc, mean_ic, icir = train_model(
        X_TRAIN, Y_TRAIN, Y_RET, label='base_y1y2', use_gpu=HAS_GPU
    )
    BASE_MODEL.save_model(BASE_MODEL_PATH)
    with open(BASE_SCALER_PATH, 'wb') as f:
        pickle.dump(BASE_SCALER, f)
    importance.to_csv(f'{RESULTS_DIR}/feature_importance_base.csv')
    elapsed = time.time() - t0
    print(f'\n  -- Base Model Results --')
    print(f'  AUC (CV mean) : {mean_auc:.4f}')
    print(f'  IC  (CV mean) : {mean_ic:.4f}')
    print(f'  ICIR          : {icir:.4f}')
    print(f'  Elapsed       : {elapsed:.1f}s')
    print(f'\n  Top 10 features:')
    print(importance.head(10).to_string())
    train_summary = {
        'model': 'base_y1y2', 'n_rows': int(len(X_TRAIN)),
        'n_features': len(FEATURE_COLS), 'mean_auc': round(mean_auc, 5),
        'mean_ic': round(mean_ic, 5), 'icir': round(icir, 5),
        'use_gpu': HAS_GPU, 'elapsed_s': round(elapsed, 1),
    }
    with open(f'{RESULTS_DIR}/train_summary.json', 'w') as f:
        json.dump(train_summary, f, indent=2)
    print(f'\n  Saved: {BASE_MODEL_PATH}')

# --- Meta-labeling model (AFML Ch. 3) ---
META_MODEL  = None
META_SCALER = None

if Path(META_MODEL_PATH).exists() and Path(META_SCALER_PATH).exists():
    print('[Cell 9] Loading cached meta model...')
    with open(META_MODEL_PATH, 'rb') as f:
        META_MODEL = pickle.load(f)
    with open(META_SCALER_PATH, 'rb') as f:
        META_SCALER = pickle.load(f)
    print(f'  Loaded: {META_MODEL_PATH}')
else:
    print('[Cell 9] Training meta-labeling model...')
    t0 = time.time()
    META_MODEL, META_SCALER = train_meta_model(
        BASE_MODEL, BASE_SCALER, X_TRAIN, Y_TRAIN,
        label='meta_base', use_gpu=HAS_GPU
    )
    if META_MODEL is not None:
        with open(META_MODEL_PATH, 'wb') as f:
            pickle.dump(META_MODEL, f)
        with open(META_SCALER_PATH, 'wb') as f:
            pickle.dump(META_SCALER, f)
        print(f'  Meta model trained in {time.time()-t0:.1f}s')
    else:
        print('  Meta model skipped (insufficient OOS data)')

gc.collect()

In [ ]:
# -- Cell 10: Walk-Forward Year 3 (Position-Tracked Fees + Meta-Labeling) -----
print('[Cell 10] Starting Year 3 walk-forward loop...')
print(f'  Period  : {YEAR3_START.date()} to {global_max.date()}')
print(f'  Retrain : every {RETRAIN_WEEKS} weeks (quarterly)')
print(f'  Universe: {ALL_DATA["symbol"].nunique()} symbols')
print(f'  Meta-labeling: {"enabled" if META_MODEL is not None else "disabled"}')
print()

weeks = pd.date_range(start=YEAR3_START, end=global_max, freq='W-MON')
if len(weeks) < 2:
    raise RuntimeError('Not enough data in Year 3. Check date splits.')

current_model  = BASE_MODEL
current_scaler = BASE_SCALER
meta_model     = META_MODEL
meta_scaler    = META_SCALER
retrain_count  = 0

all_trades_list     = []
weekly_summary_list = []
weekly_returns_hist = []
prev_longs  = set()
prev_shorts = set()
ic_history  = []  # Rolling IC for IC-weighted adjustment (Grinold & Kahn)

for week_num, (ws, we) in enumerate(zip(weeks[:-1], weeks[1:]), 1):

    # 1. Slice this week's data
    week_df = ALL_DATA[(ALL_DATA.index >= ws) & (ALL_DATA.index < we)].copy()
    if len(week_df) < 10:
        continue

    # 2. Predict (with meta-labeling confidence)
    for c_col in FEATURE_COLS:
        if c_col not in week_df.columns: week_df[c_col] = 0.0
    feat_w  = week_df[FEATURE_COLS].values.astype(np.float32)
    valid_w = np.isfinite(feat_w).all(axis=1)
    if valid_w.sum() < 5:
        continue
    try:
        feat_scaled = current_scaler.transform(feat_w[valid_w])
        probs_w     = current_model.predict_proba(feat_scaled)[:, 1]
    except Exception as e:
        print(f'  Week {week_num}: predict failed -- {e}')
        continue

    valid_idx  = np.where(valid_w)[0]
    week_valid = week_df.iloc[valid_idx].copy()
    week_valid['prob'] = probs_w

    # Meta-labeling confidence per bar
    meta_conf_arr = np.ones(len(probs_w), dtype=np.float32)
    if meta_model is not None and meta_scaler is not None:
        try:
            meta_input  = np.column_stack([feat_scaled, probs_w.reshape(-1, 1)])
            meta_scaled = meta_scaler.transform(meta_input)
            meta_conf_arr = meta_model.predict_proba(meta_scaled)[:, 1]
        except Exception:
            pass
    week_valid['meta_confidence'] = meta_conf_arr

    # 3. Cross-sectional rank signals per timestamp
    def rank_cs(grp):
        n = max(1, int(len(grp) * TOP_QUANTILE))
        grp = grp.copy().sort_values('prob')
        grp['signal'] = 'HOLD'
        if len(grp) >= 4:
            grp.iloc[-n:, grp.columns.get_loc('signal')] = 'BUY'
            grp.iloc[:n,  grp.columns.get_loc('signal')] = 'SELL'
        return grp
    week_valid = week_valid.groupby(week_valid.index, group_keys=False).apply(rank_cs)

    # 4. Aggregate symbol-level signals
    sym_signals = {}
    for _, row in week_valid[week_valid['signal'] != 'HOLD'].iterrows():
        sym = row.get('symbol', '')
        if sym not in sym_signals:
            sym_signals[sym] = {'signal': row['signal'], 'prob': [], 'ret': [], 'meta': []}
        sym_signals[sym]['prob'].append(float(row['prob']))
        fret = row.get('future_ret', np.nan)
        if np.isfinite(fret):
            sym_signals[sym]['ret'].append(float(fret))
        sym_signals[sym]['meta'].append(float(row['meta_confidence']))

    # 5. Position-tracked fee simulation
    cur_longs  = set()
    cur_shorts = set()
    trades_this_week = []

    for sym, info in sym_signals.items():
        avg_prob = float(np.mean(info['prob']))
        avg_ret  = float(np.mean(info['ret'])) if info['ret'] else 0.0
        avg_meta = float(np.mean(info['meta']))

        if info['signal'] == 'BUY':
            fee = 0.0 if sym in prev_longs else ROUND_TRIP_FEE
            pnl = (avg_ret - fee) * avg_meta * 100
            cur_longs.add(sym)
        else:
            fee = 0.0 if sym in prev_shorts else ROUND_TRIP_FEE
            pnl = (-avg_ret - fee) * avg_meta * 100
            cur_shorts.add(sym)

        trades_this_week.append({
            'week': week_num, 'week_start': str(ws.date()),
            'symbol': sym, 'signal': info['signal'],
            'pred_prob': round(avg_prob, 5),
            'pnl_percent': round(float(pnl), 4),
            'raw_ret_pct': round(avg_ret * 100, 4),
            'meta_size': round(avg_meta, 4),
        })
    all_trades_list.extend(trades_this_week)

    # 6. Confidence-weighted weekly return
    if trades_this_week:
        sizes = np.array([t['meta_size'] for t in trades_this_week])
        pnls  = np.array([t['pnl_percent'] for t in trades_this_week])
        week_ret = float(np.average(pnls, weights=sizes)) / 100
    else:
        week_ret = 0.0
    weekly_returns_hist.append(week_ret)

    # 7. IC + IC-weighted adjustment (Grinold & Kahn)
    cs_ic = compute_ic(week_valid['prob'].values,
                       week_valid['future_ret'].fillna(0).values) if 'future_ret' in week_valid.columns else 0.0
    ic_history.append(cs_ic)
    # Rolling 13-week IC quality → ic_adjusted multiplier
    lookback = min(13, len(ic_history))
    ic_adjusted = 1.0 + 10.0 * np.mean(ic_history[-lookback:])
    ic_adjusted = max(0.1, min(3.0, ic_adjusted))  # clamp [0.1, 3.0]
    ann_proj = ((1 + week_ret) ** 52 - 1) * 100

    # 8. Quarterly retrain (primary + meta model)
    did_retrain = False
    if week_num % RETRAIN_WEEKS == 0:
        print(f'  Week {week_num:3d}: QUARTERLY RETRAIN...')
        try:
            rt_df = ALL_DATA[ALL_DATA.index < we].copy()
            for c_col in FEATURE_COLS:
                if c_col not in rt_df.columns: rt_df[c_col] = 0.0
            if 'alpha_label' not in rt_df.columns:
                rt_df['alpha_label'] = rt_df.groupby(rt_df.index)['future_ret'].transform(lambda x: (x > x.median()).astype(float))
            Xr = rt_df[FEATURE_COLS].values.astype(np.float32)
            yr = rt_df['alpha_label'].values.astype(np.float32)
            yr_r = rt_df['future_ret'].values.astype(np.float32) if 'future_ret' in rt_df.columns else None
            vm = np.isfinite(Xr).all(axis=1) & np.isfinite(yr)
            Xr, yr = Xr[vm], yr[vm]
            if yr_r is not None: yr_r = yr_r[vm]
            if len(Xr) > MAX_TRAIN_ROWS:
                idx2 = np.random.choice(len(Xr), MAX_TRAIN_ROWS, replace=False)
                Xr, yr = Xr[idx2], yr[idx2]
                if yr_r is not None: yr_r = yr_r[idx2]
            m_new, s_new, imp_new, auc_n, ic_n, icir_n = train_model(Xr, yr, yr_r, label=f'y3_w{week_num:03d}', use_gpu=HAS_GPU)
            current_model = m_new; current_scaler = s_new
            retrain_count += 1
            m_new.save_model(f'{RESULTS_DIR}/models/model_y3_week{week_num:03d}.json')
            imp_new.to_csv(f'{RESULTS_DIR}/feature_importance_y3_week{week_num:03d}.csv')
            print(f'    Primary: AUC={auc_n:.4f}  IC={ic_n:.4f}  ICIR={icir_n:.4f}')

            # Retrain meta model alongside primary
            meta_new, meta_scaler_new = train_meta_model(
                current_model, current_scaler, Xr, yr,
                label=f'meta_w{week_num:03d}', use_gpu=HAS_GPU
            )
            if meta_new is not None:
                meta_model, meta_scaler = meta_new, meta_scaler_new
                with open(f'{RESULTS_DIR}/models/meta_model_week{week_num:03d}.pkl', 'wb') as mf:
                    pickle.dump(meta_model, mf)

            did_retrain = True
            del rt_df, Xr, yr; gc.collect()
        except Exception as e:
            print(f'    Retrain failed (keeping current model): {e}')

    # 9. Turnover tracking
    n_cur = len(cur_longs) + len(cur_shorts)
    n_new = len(cur_longs - prev_longs) + len(cur_shorts - prev_shorts)
    turnover_pct = round(n_new / n_cur * 100, 1) if n_cur > 0 else 100.0
    prev_longs, prev_shorts = cur_longs, cur_shorts

    # 10. Log
    on_track = week_ret >= ((1.10) ** (1/52) - 1)
    weekly_summary_list.append({
        'week': week_num, 'week_start': str(ws.date()), 'week_end': str(we.date()),
        'n_symbols': int(week_valid['symbol'].nunique()) if 'symbol' in week_valid.columns else 0,
        'n_trades': len(trades_this_week),
        'week_return_pct': round(week_ret * 100, 4),
        'annualised_pct': round(ann_proj, 2),
        'ic': round(cs_ic, 5),
        'turnover_pct': turnover_pct,
        'on_track': on_track,
        'retrained': did_retrain,
    })

    if week_num % 4 == 0 or week_num <= 2:
        rolling = np.mean(weekly_returns_hist[-4:]) * 100 if weekly_returns_hist else 0
        print(f'  Week {week_num:3d} | ret={week_ret*100:+.2f}%  IC={cs_ic:+.4f}  4w_avg={rolling:+.2f}%  n={len(trades_this_week)}  TO={turnover_pct:.0f}%')

print(f'\n[Cell 10] Walk-forward complete.')
print(f'  Total weeks : {len(weekly_summary_list)}')
print(f'  Total trades: {len(all_trades_list)}')
print(f'  Retrains    : {retrain_count}')
gc.collect()

In [ ]:
# -- Cell 11: Save Results & Performance Metrics (v2.1) ----------------------
print('[Cell 11] Saving results...')

if not weekly_summary_list:
    print('[WARN] No weekly results. Did Cell 10 run successfully?')
else:
    weekly_df = pd.DataFrame(weekly_summary_list)
    trades_df = pd.DataFrame(all_trades_list)

    weekly_df.to_csv(f'{RESULTS_DIR}/weekly_summary_year3.csv', index=False)
    trades_df.to_csv(f'{RESULTS_DIR}/all_trades_year3.csv', index=False)
    print(f'  Weekly summary -> {RESULTS_DIR}/weekly_summary_year3.csv')
    print(f'  All trades     -> {RESULTS_DIR}/all_trades_year3.csv')

    rets  = weekly_df['week_return_pct'].dropna() / 100
    pnls  = trades_df['pnl_percent'].dropna() / 100 if len(trades_df) > 0 else pd.Series(dtype=float)
    ic_s  = weekly_df['ic'].dropna()

    cum     = (1 + rets).cumprod()
    n_wks   = max(len(rets), 1)
    t_ret   = float(cum.iloc[-1] - 1) if len(cum) > 0 else 0.0
    ann_ret = (1 + t_ret) ** (52 / n_wks) - 1
    sharpe  = float(rets.mean() / (rets.std() + 1e-9) * np.sqrt(52))
    max_dd  = float(((cum - cum.cummax()) / (1 + cum.cummax())).min()) if len(cum) > 0 else 0.0
    win_rt  = float((pnls > 0).mean() * 100) if len(pnls) > 0 else 0.0
    ic_mean = float(ic_s.mean()) if len(ic_s) > 0 else 0.0
    ic_std  = float(ic_s.std())  if len(ic_s) > 0 else 0.0
    icir    = float(ic_mean / (ic_std + 1e-8))
    ic_pos  = float((ic_s > 0).mean() * 100) if len(ic_s) > 0 else 0.0

    # Turnover stats
    avg_turnover = float(weekly_df['turnover_pct'].mean()) if 'turnover_pct' in weekly_df.columns else 0.0
    meta_used = 'meta_size' in trades_df.columns

    performance = {
        'label': 'Year3_WalkForward_v2.1',
        'total_weeks': n_wks, 'total_trades': len(trades_df), 'retrains': retrain_count,
        'total_return_pct': round(t_ret * 100, 2),
        'annualised_pct': round(ann_ret * 100, 2),
        'sharpe': round(sharpe, 4),
        'max_drawdown_pct': round(max_dd * 100, 2),
        'win_rate_pct': round(win_rt, 2),
        'weeks_on_track': int(weekly_df['on_track'].sum()),
        'ic_mean': round(ic_mean, 5),
        'ic_std': round(ic_std, 5),
        'icir': round(icir, 4),
        'ic_positive_pct': round(ic_pos, 1),
        'avg_turnover_pct': round(avg_turnover, 1),
        'meta_labeling': meta_used,
        'position_tracked_fees': True,
        'use_gpu': HAS_GPU,
    }
    with open(f'{RESULTS_DIR}/performance_year3.json', 'w') as f:
        json.dump(performance, f, indent=2)

    print(f'\n  -- Year 3 Out-of-Sample Performance (v2.1) --')
    print(f'  Total Return  : {performance["total_return_pct"]:>8.2f}%')
    print(f'  Annualised    : {performance["annualised_pct"]:>8.2f}%')
    print(f'  Sharpe        : {performance["sharpe"]:>8.4f}')
    print(f'  Max Drawdown  : {performance["max_drawdown_pct"]:>8.2f}%')
    print(f'  Win Rate      : {performance["win_rate_pct"]:>8.2f}%')
    print(f'  IC Mean       : {performance["ic_mean"]:>8.5f}')
    print(f'  ICIR          : {performance["icir"]:>8.4f}')
    print(f'  IC Positive % : {performance["ic_positive_pct"]:>8.1f}%')
    print(f'  Avg Turnover  : {performance["avg_turnover_pct"]:>8.1f}%')
    print(f'  Weeks on Track: {performance["weeks_on_track"]} / {n_wks}')
    print(f'  Retrains      : {retrain_count}')
    print(f'  Meta-labeling : {meta_used}')
    print(f'  Fee model     : position-tracked (entry-only)')

    ic_grade   = ('STRONG (>0.05)' if ic_mean > 0.05 else 'GOOD (>0.03)' if ic_mean > 0.03 else 'ACCEPTABLE (>0.01)' if ic_mean > 0.01 else 'WEAK')
    icir_grade = ('EXCELLENT (>1.0)' if icir > 1.0 else 'GOOD (>0.5)' if icir > 0.5 else 'ACCEPTABLE (>0.2)' if icir > 0.2 else 'WEAK')
    print(f'\n  IC Grade  : {ic_grade}')
    print(f'  ICIR Grade: {icir_grade}')

In [ ]:
# -- Cell 12: Charts + Package Results into ZIP (v2.1) -----------------------
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import zipfile

print('[Cell 12] Generating performance chart...')

if not weekly_summary_list:
    print('[WARN] No results to chart. Run Cells 10-11 first.')
else:
    weekly_df = pd.DataFrame(weekly_summary_list)
    trades_df = pd.DataFrame(all_trades_list)

    fig = plt.figure(figsize=(16, 11))
    fig.suptitle('Azalyst v2.1 -- Year 3 Walk-Forward (Meta-Labeling + Position-Tracked Fees)',
                 fontsize=14, fontweight='bold')
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

    # Panel 1: Cumulative return
    ax1 = fig.add_subplot(gs[0, 0])
    rets = weekly_df['week_return_pct'].fillna(0) / 100
    cum  = ((1 + rets).cumprod() - 1) * 100
    ax1.plot(weekly_df['week'], cum, color='#1f77b4', linewidth=2)
    ax1.fill_between(weekly_df['week'], cum, alpha=0.12, color='#1f77b4')
    ax1.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax1.set_title('Cumulative Return (%)', fontweight='bold')
    ax1.set_xlabel('Week #'); ax1.set_ylabel('%'); ax1.grid(True, alpha=0.25)

    # Panel 2: Weekly return distribution
    ax2 = fig.add_subplot(gs[0, 1])
    wr = weekly_df['week_return_pct'].dropna()
    ax2.hist(wr, bins=min(30, max(10, len(wr)//3)), color='#ff7f0e', alpha=0.72, edgecolor='black', linewidth=0.4)
    if len(wr) > 2:
        ax2.axvline(wr.mean(),   color='red',   linewidth=1.8, linestyle='--', label=f'Mean {wr.mean():.2f}%')
        ax2.axvline(wr.median(), color='green', linewidth=1.2, linestyle=':',  label=f'Median {wr.median():.2f}%')
        ax2.legend(fontsize=9)
    ax2.set_title('Weekly Return Distribution', fontweight='bold')
    ax2.set_xlabel('Weekly Return (%)'); ax2.set_ylabel('Count'); ax2.grid(True, alpha=0.25)

    # Panel 3: IC series
    ax3 = fig.add_subplot(gs[1, 0])
    ic_vals = weekly_df['ic'].fillna(0)
    ax3.bar(weekly_df['week'], ic_vals, color=['#2ca02c' if v > 0 else '#d62728' for v in ic_vals], alpha=0.75, width=0.8)
    if len(ic_vals) > 2:
        ax3.axhline(ic_vals.mean(), color='navy', linewidth=1.5, linestyle='--', label=f'Mean IC {ic_vals.mean():.4f}')
        ax3.legend(fontsize=9)
    ax3.axhline(0, color='black', linewidth=0.6)
    ax3.set_title('Weekly IC (Information Coefficient)', fontweight='bold')
    ax3.set_xlabel('Week #'); ax3.set_ylabel('Spearman IC'); ax3.grid(True, alpha=0.25)

    # Panel 4: Trade P&L distribution
    ax4 = fig.add_subplot(gs[1, 1])
    if len(trades_df) > 0 and 'pnl_percent' in trades_df.columns:
        pnl = trades_df['pnl_percent'].dropna()
        ax4.hist(pnl, bins=min(40, max(10, len(pnl)//20)), color='#9467bd', alpha=0.72, edgecolor='black', linewidth=0.3)
        ax4.axvline(pnl.mean(), color='red', linewidth=1.8, linestyle='--', label=f'Mean {pnl.mean():.3f}%')
        ax4.axvline(0, color='black', linewidth=0.8)
        ax4.legend(fontsize=9)
        ax4.set_title(f'Trade P&L Distribution  (n={len(pnl):,})', fontweight='bold')
        ax4.set_xlabel('P&L per Trade (%)'); ax4.set_ylabel('Count'); ax4.grid(True, alpha=0.25)

    chart_path = f'{RESULTS_DIR}/performance_year3.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Chart saved -> {chart_path}')

    # Zip everything
    zip_path = f'{RESULTS_DIR}/azalyst_v2_results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fp in Path(RESULTS_DIR).rglob('*'):
            if fp.is_file() and fp.suffix in ('.csv', '.json', '.png', '.pkl'):
                zf.write(fp, fp.relative_to(Path(RESULTS_DIR).parent))
    zip_mb = Path(zip_path).stat().st_size / 1e6
    print(f'  Results zip -> {zip_path}  ({zip_mb:.1f} MB)')

    print(f'''
=========================================================
  AZALYST v2.1 -- RUN COMPLETE
  Built by Azalyst Research
=========================================================
  Total Return   : {performance["total_return_pct"]:.2f}%
  Annualised     : {performance["annualised_pct"]:.2f}%
  Sharpe         : {performance["sharpe"]:.4f}
  IC Mean        : {performance["ic_mean"]:.5f}
  ICIR           : {performance["icir"]:.4f}
  Meta-labeling  : {performance["meta_labeling"]}
  Fee model      : position-tracked (entry-only)
  Avg Turnover   : {performance["avg_turnover_pct"]:.1f}%
  GPU used       : {HAS_GPU}
=========================================================
  Three Pillars of Alpha (v2.1):
    1. Fractional Differentiation (AFML Ch. 5)
    2. Meta-Labeling Confidence Sizing (AFML Ch. 3)
    3. IC-Weighted Signal Combination (Grinold & Kahn)
=========================================================
  Download azalyst_v2_results.zip from the Output tab.
  Send performance_year3.json + weekly_summary_year3.csv
  to Claude for interpretation (README grading table).
''')